# Ch 07 · A Guided Tour of step and forward

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/07-pipeline-tour.ipynb)

Companion notebook for the chapter page `chapters/07-pipeline-tour.html` ([page](../chapters/07-pipeline-tour.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

This notebook holds every example of Chapter 7 in page order. All examples are device-agnostic: on a Colab GPU they use the big budgets (`nworld=4096`, 10,240 balls in the checkpoint), on CPU the small ones. Numbers you see here are **your** numbers; the chapter page shows CPU runs.

## 2. Inside `forward`: calling the stages yourself
`mjw.forward` runs `fwd_position`, `fwd_velocity`, `fwd_actuation`, `fwd_acceleration` and `solve` (plus sensors). Example 7.1 reproduces it with the public stage functions and their default factorize pairing; Example 7.2 prints what each stage fills in.

In [ ]:
# Example 7.1 - ex1_manual_stages.py
"""Example 7.1 - call the forward stages by hand and reproduce mjw.forward exactly.

The public stage functions use the "factorize in the position stage" pairing:
fwd_position(factorize=True) writes qLD, then fwd_acceleration(factorize=False) solves with it.
mjw.forward uses the other pairing internally (factorize=False, then a fused factor+solve).
Both give the same qacc. Never combine fwd_position(factorize=False) with the default
fwd_acceleration(): that solves with an old factor (Ch 11 owns that experiment).
Run: python ex1_manual_stages.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50
SETTLE = 150                    # 1.5 s: the ball and box land, so there are contacts

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)

# a different motor command per world, then let the objects land so there are contacts
d.ctrl.assign(np.linspace(-1.0, 1.0, NWORLD, dtype=np.float32).reshape(NWORLD, 1))
for _ in range(SETTLE):
    mjw.step(m, d)

# 1) the reference: one call
mjw.forward(m, d)
qacc_ref = d.qacc.numpy().copy()

# 2) the same thing, stage by stage (public defaults: factor in the position stage)
mjw.fwd_position(m, d)          # factorize=True: kinematics ... crb, factor_m, collision, constraints
mjw.fwd_velocity(m, d)          # com_vel, passive, rne -> qfrc_bias
mjw.fwd_actuation(m, d)         # actuator_force -> qfrc_actuator
mjw.fwd_acceleration(m, d)      # factorize=False: qfrc_smooth, solve with qLD -> qacc_smooth
mjw.solve(m, d)                 # constraint solver -> qacc
qacc_manual = d.qacc.numpy()

print("device:", wp.get_device(), "| nworld:", NWORLD, "| nv:", mjm.nv)
print("contacts in the pool (all worlds):", int(d.nacon.numpy()[0]))
print("max |qacc(forward) - qacc(manual stages)|:", float(np.abs(qacc_ref - qacc_manual).max()))
print("world 0 qacc:", qacc_ref[0])
print("world 0 qacc_smooth (before the constraint solve):", d.qacc_smooth.numpy()[0])
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

In [ ]:
# Example 7.2 - ex2_stage_outputs.py
"""Example 7.2 - run the stages one at a time and print what each one fills in.

A fresh Data starts with zeros in every derived field. We copy a settled state (qpos, qvel, ctrl)
into it and call the five stages in order, printing a few outputs after each one.
Run: python ex2_stage_outputs.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50
SETTLE = 150                    # 1.5 s: the ball and box land, so there are contacts

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
run = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
run.ctrl.assign(np.linspace(-1.0, 1.0, NWORLD, dtype=np.float32).reshape(NWORLD, 1))
for _ in range(SETTLE):
    mjw.step(m, run)

# a fresh Data: copy only the state and the inputs; every derived field is still zero
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
wp.copy(d.qpos, run.qpos)
wp.copy(d.qvel, run.qvel)
wp.copy(d.ctrl, run.ctrl)
W = 0      # world to print
BALL, BOX, ARM = 1, 2, 3
HINGE = 12  # dof index of the arm hinge (6 ball dofs + 6 box dofs come first)


def show(name, arr):
    print(f"  {name:<22}", arr)


print("before any stage: xpos[ball] =", d.xpos.numpy()[W, BALL], "| nacon =", d.nacon.numpy()[0])

mjw.fwd_position(m, d)
print("after fwd_position (kinematics, com, crb + factor_m, collision, constraints, transmission)")
show("xpos[ball]", d.xpos.numpy()[W, BALL])
show("xpos[box]", d.xpos.numpy()[W, BOX])
show("M nonzeros (of nC=%d)" % mjm.nC, int(np.count_nonzero(d.M.numpy()[W])))
show("M diagonal[ball]", d.M.numpy()[W, :6])
show("nacon (all worlds)", int(d.nacon.numpy()[0]))
show("nefc (world 0)", int(d.nefc.numpy()[W]))
show("actuator_moment", d.actuator_moment.numpy()[W])

mjw.fwd_velocity(m, d)
print("after fwd_velocity (com_vel, passive, rne)")
show("cvel[arm] (rot, lin)", d.cvel.numpy()[W, ARM])
show("qfrc_bias", d.qfrc_bias.numpy()[W])

mjw.fwd_actuation(m, d)
print("after fwd_actuation")
show("ctrl", d.ctrl.numpy()[W])
show("actuator_force", d.actuator_force.numpy()[W])
show("qfrc_actuator[hinge]", d.qfrc_actuator.numpy()[W, HINGE])

mjw.fwd_acceleration(m, d)
print("after fwd_acceleration (sum forces, solve M qacc_smooth = qfrc_smooth)")
show("qfrc_smooth", d.qfrc_smooth.numpy()[W])
show("qacc_smooth", d.qacc_smooth.numpy()[W])

mjw.solve(m, d)
print("after solve (constraint forces)")
show("qacc", d.qacc.numpy()[W])
show("qfrc_constraint", d.qfrc_constraint.numpy()[W])
show("solver iterations", int(d.solver_niter.numpy()[W]))

mjw.forward(m, run)
print("same qacc as mjw.forward:", bool(np.allclose(d.qacc.numpy(), run.qacc.numpy(), atol=1e-5)))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 4. The stage → Data cheat sheet, measured
Example 7.3 wraps `wp.launch` / `wp.launch_tiled` with `unittest.mock` to see which `Data` arrays each stage passes as kernel inputs and outputs, and compares every field before and after.

In [ ]:
# Example 7.3 - ex3_stage_fields.py
"""Example 7.3 - measure which Data fields each stage reads and writes.

For every stage we (a) record the Data arrays passed as kernel inputs and outputs by wrapping
wp.launch / wp.launch_tiled with unittest.mock, and (b) compare every Data field before and after
the stage. This is a measured version of the stage -> Data cheat sheet.
Run: python ex3_stage_fields.py [--dump-json PATH]
  --dump-json PATH writes {stages: [{name, reads, writes, changed}]} for the page's interactive sheet.
"""
import argparse
import json
from unittest import mock

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50
SETTLE = 150

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
run = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
run.ctrl.assign(np.linspace(-1.0, 1.0, NWORLD, dtype=np.float32).reshape(NWORLD, 1))
for _ in range(SETTLE):
    mjw.step(m, run)

d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)   # derived fields all zero
for name in ("qpos", "qvel", "ctrl"):
    wp.copy(getattr(d, name), getattr(run, name))


def data_fields(obj, prefix=""):
    """Map every wp.array in Data (and d.contact, d.efc) to its name."""
    out = {}
    for name, val in vars(obj).items():
        if isinstance(val, wp.array):
            out[prefix + name] = val
        elif name in ("contact", "efc"):
            out.update(data_fields(val, prefix + name + "."))
    return out


FIELDS = data_fields(d)
BY_PTR = {}
for name, arr in FIELDS.items():
    if arr.ptr is not None:
        BY_PTR.setdefault(arr.ptr, name)


def names(arrs):
    return {BY_PTR[a.ptr] for a in arrs if isinstance(a, wp.array) and a.ptr in BY_PTR}


def snapshot():
    return {k: v.numpy().copy() for k, v in FIELDS.items() if v.size}


STAGES = [  # public functions, in the order forward runs them (public factorize pairing)
    ("kinematics", mjw.kinematics), ("com_pos", mjw.com_pos), ("camlight", mjw.camlight),
    ("flex", mjw.flex), ("tendon", mjw.tendon), ("crb", mjw.crb), ("factor_m", mjw.factor_m),
    ("collision", mjw.collision), ("make_constraint", mjw.make_constraint),
    ("transmission", mjw.transmission), ("sensor_pos", mjw.sensor_pos),
    ("fwd_velocity", mjw.fwd_velocity), ("sensor_vel", mjw.sensor_vel),
    ("fwd_actuation", mjw.fwd_actuation), ("fwd_acceleration", mjw.fwd_acceleration),
    ("solve", mjw.solve), ("sensor_acc", mjw.sensor_acc), ("euler", mjw.euler),
]

real_launch, real_tiled = wp.launch, wp.launch_tiled
rows = []
for stage, fn in STAGES:
    reads, writes, nlaunch = set(), set(), [0]

    def spy(real):
        def wrapper(kernel, *a, **kw):
            nlaunch[0] += 1
            pos = a if "dim" in kw else a[1:]          # positional args after dim
            reads.update(names(kw.get("inputs", pos[0] if pos else [])))
            writes.update(names(kw.get("outputs", pos[1] if len(pos) > 1 else [])))
            return real(kernel, *a, **kw)
        return wrapper

    before = snapshot()
    with mock.patch.object(wp, "launch", spy(real_launch)), \
         mock.patch.object(wp, "launch_tiled", spy(real_tiled)):
        fn(m, d)
    after = snapshot()
    changed = sorted(k for k in after if not np.array_equal(before[k], after[k]))
    rows.append({"name": stage, "launches": nlaunch[0], "reads": sorted(reads - writes),
                 "writes": sorted(writes | set(changed)), "changed": changed})

print(f"PLAYGROUND, nworld={NWORLD}: Data fields per stage (reads = kernel inputs only)")
for r in rows:
    w = ", ".join(r["changed"]) or "-"
    print(f"{r['name']:<17} launches {r['launches']:>2} | changed: {w}")
print("fields written but unchanged (zero in this scene), e.g. fwd_velocity:",
      ", ".join(sorted(set(rows[11]["writes"]) - set(rows[11]["changed"]))[:6]))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"model": "PLAYGROUND", "nworld": NWORLD, "stages": rows}, f,
                  separators=(",", ":"))
    print("wrote", args.dump_json, len(rows), "stages")

## 5. The force budget
`qfrc_smooth = qfrc_passive - qfrc_bias + qfrc_actuator + qfrc_applied (+ J^T xfrc_applied)`. Example 7.4 checks the identity per world; Example 7.9 pushes the box through `xfrc_applied`.

In [ ]:
# Example 7.4 - ex4_force_budget.py
"""Example 7.4 - audit the force budget: qfrc_smooth is a sum of named force terms.

fwd_acceleration computes, per world and per dof,
  qfrc_smooth = qfrc_passive - qfrc_bias + qfrc_actuator + qfrc_applied  (+ J^T xfrc_applied)
(forward.py::_qfrc_smooth, then support.py::xfrc_accumulate). We rebuild the right-hand side
from the individual Data fields and print the residual. The physics of each term is Ch 11-12.
Run: python ex4_force_budget.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)

rng = np.random.default_rng(0)
# give every term something to do:
d.ctrl.assign(rng.uniform(-1, 1, (NWORLD, mjm.nu)).astype(np.float32))         # actuator
d.qvel.assign(rng.normal(0, 1, (NWORLD, mjm.nv)).astype(np.float32))           # bias + damper
qfrc_applied = np.zeros((NWORLD, mjm.nv), np.float32)
qfrc_applied[:, 0] = rng.uniform(-2, 2, NWORLD)                                # push the ball in x
d.qfrc_applied.assign(qfrc_applied)
xfrc = np.zeros((NWORLD, mjm.nbody, 6), np.float32)
xfrc[:, 2, 0] = 5.0                                                            # 5 N on the box, +x
xfrc[:, 2, 5] = 0.5                                                            # 0.5 N m about z
d.xfrc_applied.assign(xfrc)
damping = m.dof_damping.numpy()
damping[:, 12] = 0.2                                                           # hinge damping
m.dof_damping.assign(damping)                                                  # array field: live

for _ in range(NSTEP):
    mjw.step(m, d)
mjw.forward(m, d)

passive, bias = d.qfrc_passive.numpy(), d.qfrc_bias.numpy()
actuator, applied = d.qfrc_actuator.numpy(), d.qfrc_applied.numpy()
jt_xfrc_arr = wp.zeros((NWORLD, mjm.nv), dtype=float)
mjw.xfrc_accumulate(m, d, jt_xfrc_arr)                                         # J^T xfrc_applied
jt_xfrc = jt_xfrc_arr.numpy()
smooth = d.qfrc_smooth.numpy()

rhs = passive - bias + actuator + applied + jt_xfrc
res = np.abs(smooth - rhs).max(axis=1)
parts = (d.qfrc_spring.numpy() + d.qfrc_damper.numpy() + d.qfrc_gravcomp.numpy()
         + d.qfrc_fluid.numpy() + d.qfrc_adhesion.numpy())
print("residual |qfrc_smooth - sum of terms| per world:", res)
print("max residual over all worlds:", f"{res.max():.2e}")
print("qfrc_passive == spring + damper + gravcomp + fluid + adhesion:",
      bool(np.allclose(passive, parts, atol=1e-6)))

W = 1
names = ["ball x", "ball y", "ball z", "ball rx", "ball ry", "ball rz",
         "box x", "box y", "box z", "box rx", "box ry", "box rz", "hinge"]
print(f"world {W}, per dof:")
print(f"{'dof':<8}{'passive':>9}{'-bias':>9}{'actuator':>9}{'applied':>9}{'J^T xfrc':>9}"
      f"{'= smooth':>10}")
for i, n in enumerate(names):
    print(f"{n:<8}{passive[W, i]:9.3f}{-bias[W, i]:9.3f}{actuator[W, i]:9.3f}"
          f"{applied[W, i]:9.3f}{jt_xfrc[W, i]:9.3f}{smooth[W, i]:10.3f}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    out = {"dofs": names, "worlds": []}
    for w in range(min(NWORLD, 4)):
        out["worlds"].append({k: [round(float(v), 3) for v in a[w]] for k, a in
                              (("passive", passive), ("neg_bias", -bias), ("actuator", actuator),
                               ("applied", applied), ("jt_xfrc", jt_xfrc), ("smooth", smooth))})
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Example 7.9 - ex9_xfrc_push.py
"""Example 7.9 - push the box with d.xfrc_applied and follow the force through the pipeline.

xfrc_applied is a Cartesian wrench per body, (nworld, nbody) spatial vectors laid out as
(force x, y, z, torque x, y, z), applied at the body's center of mass. fwd_acceleration maps it to
joint space with J^T (support.py::xfrc_accumulate) and adds it to qfrc_smooth. Every world pushes
with a different force. We lower the sliding friction of all geoms to 0.3 at runtime (geom_friction
is an array field, so the change is live), so boxes slide above mu m g = 23.5 N and only tip over
above m g = 78.5 N.
Run: python ex9_xfrc_push.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50
SETTLE = 100                     # the box lands first
PUSH = 50                        # then 0.5 s of pushing

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
BOX = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "box")
BOX_DOF = mjm.jnt_dofadr[mjm.body_jntadr[BOX]]           # first of the box's 6 dofs
print("box body id:", BOX, "| box dofs:", BOX_DOF, "..", BOX_DOF + 5,
      "| box mass:", round(float(mjm.body_mass[BOX]), 3), "kg")

MU = 0.3
fr = m.geom_friction.numpy()                                # (1, ngeom, 3): slide, spin, roll
fr[..., 0] = MU
m.geom_friction.assign(fr)

for _ in range(SETTLE):
    mjw.step(m, d)

force = np.linspace(0.0, 70.0, NWORLD, dtype=np.float32)    # newtons, one value per world
xfrc = np.zeros((NWORLD, mjm.nbody, 6), np.float32)
xfrc[:, BOX, 0] = force                                     # +x force on the box
d.xfrc_applied.assign(xfrc)                                 # assign: keeps the same array

x0 = d.qpos.numpy()[:, 7].copy()                            # box x position (qpos 7..13)
for _ in range(PUSH):
    mjw.step(m, d)
mjw.forward(m, d)                                           # refresh qfrc_smooth for printing

qs = d.qfrc_smooth.numpy()[:, BOX_DOF]
vx = d.qvel.numpy()[:, BOX_DOF]
dx = d.qpos.numpy()[:, 7] - x0
mass = float(mjm.body_mass[BOX])
t_push = PUSH * float(mjm.opt.timestep)
coulomb = np.maximum(0.0, (force - MU * mass * 9.81) / mass * t_push)   # rigid Coulomb estimate
print(f"{'world':>5}{'F [N]':>8}{'qfrc_smooth x':>15}{'vx [m/s]':>10}{'Coulomb vx':>12}{'dx [m]':>9}")
for w in range(NWORLD):
    print(f"{w:5d}{force[w]:8.1f}{qs[w]:15.3f}{vx[w]:10.3f}{coulomb[w]:12.3f}{dx[w]:9.3f}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 7. Disable flags
`m.opt.disableflags` is a Python int read at dispatch: flip bits between steps (and recapture any CUDA graph afterwards).

In [ ]:
# Example 7.7 - ex7_disable_flags.py
"""Example 7.7 - switch pipeline stages off with disable flags and watch the effect.

m.opt.disableflags is a Python int read when the step is dispatched, so you can change it between
steps (a change means different kernels run; a captured CUDA graph must be recaptured, Ch 24).
We run the same 100 steps with nothing disabled, then with GRAVITY, CONTACT and ACTUATION off.
Run: python ex7_disable_flags.py [--dump-json PATH]
  --dump-json PATH writes world 0's ball z, box z and hinge angle every 5 steps, per setting.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 100                     # 1 s at timestep 0.01

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
HINGE_QPOS = 14                  # ball qpos 0..6, box qpos 7..13, hinge 14

series = {}
print(f"nworld={NWORLD}, {NSTEP} steps, motor ctrl = 0.5 in every world")
print(f"{'disabled':<12}{'ball z':>9}{'box z':>9}{'hinge q':>10}{'nacon':>7}{'nefc[0]':>9}")
for label, flag in (("nothing", 0), ("GRAVITY", mjw.DisableBit.GRAVITY),
                    ("CONTACT", mjw.DisableBit.CONTACT), ("ACTUATION", mjw.DisableBit.ACTUATION),
                    ("CONSTRAINT", mjw.DisableBit.CONSTRAINT)):
    m.opt.disableflags = int(flag)
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
    d.ctrl.fill_(0.5)
    frames = []
    for i in range(NSTEP):
        mjw.step(m, d)
        if args.dump_json and i % 5 == 4:          # host read for the figure only
            q0 = d.qpos.numpy()[0]
            frames.append([round(float(q0[2]), 3), round(float(q0[9]), 3),
                           round(float(q0[HINGE_QPOS]), 3)])
    series[label] = frames
    q = d.qpos.numpy()
    print(f"{label:<12}{q[0, 2]:9.3f}{q[0, 9]:9.3f}{q[0, HINGE_QPOS]:10.3f}"
          f"{int(d.nacon.numpy()[0]):7d}{int(d.nefc.numpy()[0]):9d}")
    assert not d.overflow.numpy().any()
m.opt.disableflags = 0
print("all worlds identical:", bool(np.allclose(q, q[0])))
print("overflow: none")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"dt": 5 * float(mjm.opt.timestep), "fields": ["ball z", "box z", "hinge q"],
                   "series": series}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 8. How many kernels, and what shape
Example 7.5 counts the launches of one step and sorts them into the launch-shape atlas layouts; Example 7.6 lists them in order with their pipeline station. On a GPU the counts are the same as on CPU, only the dims grow with `nworld`.

In [ ]:
# Example 7.5 - ex5_count_launches.py
"""Example 7.5 - count the kernel launches in one mjw.step and sort them by launch shape.

wp.launch and wp.launch_tiled are wrapped with unittest.mock (the real functions still run), so
we see every launch: kernel name, the calling source line, and the launch `dim`. Each launch is
then put into one of the pipeline-wide launch-shape layouts of the atlas:
  1 (world, element) grid     dim=(nworld, n)          e.g. smooth.py kinematics
  2 flat pool slots           dim=naconmax             e.g. collision_primitive.py narrowphase
  6 tiled, block per world    wp.launch_tiled(...)     e.g. solver.py, smooth.py factor blocks
  7 3-D grid                  dim=(nworld, n, k)       e.g. solver.py jaref / jv
  8 row per contact           dim=naconmax in constraint.py
Run: python ex5_count_launches.py [--dump-json PATH]
"""
import argparse
import collections
import json
import os
import sys
from unittest import mock

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50
SETTLE = 30                     # a few steps first, so contacts and constraints exist

HUMANOID_PATH = os.path.join(os.path.dirname(mjw.__file__), "test_data", "humanoid", "humanoid.xml")


def layout_of(dim, tiled, file, d):
    if tiled:
        return 6
    if isinstance(dim, int):
        if dim == d.naconmax:
            return 8 if file == "constraint.py" else 2
        return 1                       # dim=nworld: a (world, 1) grid
    if len(dim) == 3:
        return 7
    return 1


def record_step(m, d):
    """Run one mjw.step and return the list of launches it made."""
    log = []

    def spy(real, tiled):
        def wrapper(kernel, *a, **kw):
            dim = kw.get("dim", a[0] if a else None)
            dim = tuple(int(x) for x in dim) if isinstance(dim, (tuple, list)) else int(dim)
            caller = sys._getframe(1)               # the MJWarp line that called wp.launch
            file = os.path.basename(caller.f_code.co_filename)
            log.append({"kernel": kernel.key.split("__locals__")[0].rstrip("_"),
                        "file": file, "line": caller.f_lineno, "func": caller.f_code.co_name,
                        "dim": dim, "tiled": tiled, "block_dim": kw.get("block_dim"),
                        "layout": layout_of(dim, tiled, file, d)})
            return real(kernel, *a, **kw)
        return wrapper

    with mock.patch.object(wp, "launch", spy(wp.launch, False)), \
         mock.patch.object(wp, "launch_tiled", spy(wp.launch_tiled, True)):
        mjw.step(m, d)
    return log


results = {}
for name, mjm in (("PLAYGROUND", mujoco.MjModel.from_xml_string(PLAYGROUND)),
                  ("HUMANOID", mujoco.MjModel.from_xml_path(HUMANOID_PATH))):
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
    for _ in range(SETTLE):
        mjw.step(m, d)
    log = record_step(m, d)
    results[name] = {"nworld": NWORLD, "naconmax": d.naconmax, "njmax": d.njmax,
                     "nv": mjm.nv, "nbody": mjm.nbody, "launches": log}
    n_plain = sum(not x["tiled"] for x in log)
    by_layout = collections.Counter(x["layout"] for x in log)
    print(f"{name}: nv={mjm.nv} nbody={mjm.nbody} naconmax={d.naconmax} njmax={d.njmax}")
    print(f"  launches in one mjw.step: {len(log)} ({n_plain} wp.launch + "
          f"{len(log) - n_plain} wp.launch_tiled), solver iterations: "
          f"{int(d.solver_niter.numpy().max())}")
    n_empty = sum(int(np.prod(x["dim"])) == 0 for x in log)
    print(f"  of these, {n_empty} have an empty dim (no tendons, flex, limits...): Warp skips them")
    print("  by layout:", dict(sorted(by_layout.items())))
    for lay in (1, 2, 6, 7, 8):
        ex = next((x for x in log if x["layout"] == lay and int(np.prod(x["dim"])) > 0), None)
        if ex:
            print(f"    layout {lay}: {ex['file']}:{ex['line']} {ex['kernel']} dim={ex['dim']}"
                  + (f" block_dim={ex['block_dim']}" if ex["tiled"] else ""))
    print("  overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(results, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Example 7.6 - ex6_kernel_names.py
"""Example 7.6 - list the kernels of one mjw.step in launch order, tagged with their stage.

Every wp.launch / wp.launch_tiled call is recorded with its kernel name and source file. Walking
up the Python call stack to the nearest pipeline function (kinematics, crb, collision, ...)
tells us which stage of the breadcrumb launched it. The order you see IS the stage order.
Run: python ex6_kernel_names.py [--dump-json PATH]
"""
import argparse
import collections
import json
import os
import sys
from unittest import mock

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50
SETTLE = 150

# Python function name in the call stack -> breadcrumb station
STATION = {
    "kinematics": "kinematics", "com_pos": "kinematics", "camlight": "kinematics",
    "flex": "kinematics", "tendon": "kinematics",
    "crb": "crb", "tendon_armature": "crb", "factor_m": "crb",
    "collision": "collision", "make_constraint": "make_constraint",
    "transmission": "transmission", "sensor_pos": "sensor_pos",
    "fwd_velocity": "velocity", "sensor_vel": "sensor_vel",
    "fwd_actuation": "actuation", "fwd_acceleration": "acceleration", "solve": "solve",
    "rne_postconstraint": "rne_post", "sensor_acc": "sensor_acc",
    "euler": "integrate", "implicit": "integrate", "rungekutta4": "integrate",
}


def station_of(frame):
    while frame is not None:
        name = frame.f_code.co_name
        if name in STATION and "mujoco_warp" in frame.f_code.co_filename:
            return STATION[name]
        frame = frame.f_back
    return "?"


mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
d.ctrl.assign(np.linspace(-1.0, 1.0, NWORLD, dtype=np.float32).reshape(NWORLD, 1))
for _ in range(SETTLE):
    mjw.step(m, d)

log = []


def spy(real):
    def wrapper(kernel, *a, **kw):
        dim = kw.get("dim", a[0] if a else None)
        size = int(np.prod(dim)) if isinstance(dim, (tuple, list)) else int(dim)
        caller = sys._getframe(1)
        log.append({"kernel": kernel.key.split("__locals__")[0].rstrip("_"),
                    "file": os.path.basename(caller.f_code.co_filename),
                    "station": station_of(caller), "empty": size == 0})
        return real(kernel, *a, **kw)
    return wrapper


with mock.patch.object(wp, "launch", spy(wp.launch)), \
     mock.patch.object(wp, "launch_tiled", spy(wp.launch_tiled)):
    mjw.step(m, d)

print(f"PLAYGROUND, one mjw.step: {len(log)} launches. First 40 in order (* = empty dim):")
for i, x in enumerate(log[:40]):
    print(f"{i:3d} {x['station']:<16}{x['kernel'] + ('*' if x['empty'] else ''):<36}{x['file']}")
counts = collections.Counter(x["station"] for x in log)
print("launches per station:", dict(counts))

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"model": "PLAYGROUND", "nworld": NWORLD, "launches": log}, f,
                  separators=(",", ":"))
    print("wrote", args.dump_json, len(log), "launches")

## 9. Where the time goes
Example 7.8 times each stage with `wp.synchronize()` around it. **On Colab these are your GPU numbers**: compare the shares with the CPU figure on the chapter page (that is the stretch exercise). Example 7.10 prints the nested `EventTracer` tree (real GPU times on a GPU, the launch-count structure on CPU).

In [ ]:
# Example 7.8 - ex8_stage_timing.py
"""Example 7.8 - time each stage by hand and print its share of the step.

Kernel launches are asynchronous on a GPU, so each stage is bracketed by wp.synchronize() before
the clock is read. The stage calls use the public factorize pairing and add up to one Euler step.
Each number is the median over REPS synchronized repeats, which is steadier than a mean on a busy
machine. These are wall-clock times on whatever device runs the script: on CPU they are CPU numbers
and say nothing about GPU performance; at small nworld they are dominated by fixed per-launch cost
and vary from run to run. On GPU, launch overhead dominates small models (see the graph
teaser in the chapter, Ch 24-25 for real profiling).
Run: python ex8_stage_timing.py [--dump-json PATH]
"""
import argparse
import json
import os
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50
REPS = 200 if ON_GPU else 20

HUMANOID_PATH = os.path.join(os.path.dirname(mjw.__file__), "test_data", "humanoid", "humanoid.xml")
STAGES = [("fwd_position", mjw.fwd_position), ("fwd_velocity", mjw.fwd_velocity),
          ("fwd_actuation", mjw.fwd_actuation), ("fwd_acceleration", mjw.fwd_acceleration),
          ("solve", mjw.solve), ("euler", mjw.euler)]

out = {"device": str(wp.get_device()), "nworld": NWORLD, "reps": REPS, "models": {}}
for name, mjm in (("PLAYGROUND", mujoco.MjModel.from_xml_string(PLAYGROUND)),
                  ("HUMANOID", mujoco.MjModel.from_xml_path(HUMANOID_PATH))):
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
    for _ in range(30):                     # warm-up: compile, settle contacts
        mjw.step(m, d)
    for s, fn in STAGES:                    # the stage path compiles different kernels
        fn(m, d)                            # (factor_m + solve_m instead of the fused solve)
    wp.synchronize()
    times = {s: [] for s, _ in STAGES}
    for _ in range(REPS):
        for s, fn in STAGES:
            t0 = time.perf_counter()
            fn(m, d)
            wp.synchronize()
            times[s].append(time.perf_counter() - t0)
    step_t = []
    for _ in range(REPS):
        t0 = time.perf_counter()
        mjw.step(m, d)
        wp.synchronize()
        step_t.append(time.perf_counter() - t0)
    step_ms = 1e3 * float(np.median(step_t))      # median of REPS synchronized steps
    stage_ms = {s: 1e3 * float(np.median(t)) for s, t in times.items()}
    sum_ms = sum(stage_ms.values())
    print(f"{name} on {wp.get_device()}, nworld={NWORLD}, median of {REPS}: mjw.step {step_ms:.2f} ms, "
          f"sum of stage medians {sum_ms:.2f} ms, {NWORLD / step_ms * 1e3:,.0f} world-steps/s")
    for s, t in stage_ms.items():
        bar = "#" * int(round(40 * t / sum_ms))
        print(f"  {s:<17}{t:7.3f} ms {100 * t / sum_ms:5.1f}%  {bar}")
    out["models"][name] = {"step_ms": round(step_ms, 3),
                           "stage_ms": {s: round(t, 3) for s, t in stage_ms.items()}}
    print("  overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Example 7.10 - ex10_event_trace.py
"""Example 7.10 - per-stage GPU timing with EventTracer (CUDA only; guarded on CPU).

Most pipeline functions carry @event_scope (warp_util.py). Inside an EventTracer they record a
CUDA event before and after the call and nest: step > forward > fwd_position > ... On CPU the first
event_scope would raise "Device cpu is not a CUDA device", so the tracer is built with
enabled=ON_GPU and the CPU trace is empty (_src/cli.py shows the enabled= switch, driven by its
--event_trace flag; the course passes ON_GPU so the tracer is off on CPU).
To still show the real *structure* on CPU, we then swap the CUDA events for a stand-in that counts
kernel launches: the same tree comes out, with "launches inside" in place of milliseconds.
Ch 25 owns the profiling workflow.
Run: python ex10_event_trace.py [--dump-json PATH]
"""
import argparse
import json
from unittest import mock

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.warp_util import EventTracer   # private API: may change between versions

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
for _ in range(100):
    mjw.step(m, d)                    # warm-up: compile outside the trace, let objects land
wp.synchronize()

with EventTracer(enabled=ON_GPU) as tracer:
    for _ in range(5):
        mjw.step(m, d)
    wp.synchronize()
    trace = tracer.trace()           # {name: ((ms per call, ...), {child: ...})}


def show(tr, unit, scale, depth=0):
    for name, (vals, sub) in tr.items():
        mean = scale * sum(vals) / len(vals)
        num = f"{mean:8.1f}" if unit == "us" else f"{int(mean):4d}"
        print(f"{'  ' * depth}{name:<{30 - 2 * depth}} calls {len(vals)} {num} {unit}")
        show(sub, unit, scale, depth + 1)


if ON_GPU:
    print(f"EventTracer on {wp.get_device()}, nworld={NWORLD}, 5 steps (your GPU numbers):")
    show(trace, "us", 1e3)            # elapsed times are milliseconds -> microseconds
else:
    print("skipped on CPU: EventTracer needs CUDA, so the trace is empty:", trace)

# Structure on any device: a stand-in "event" remembers how many launches happened so far.
COUNT = [0]


class LaunchCountEvent:
    def __init__(self, *a, **kw):
        self.n = 0


def record(ev, *a, **kw):
    ev.n = COUNT[0]


def counting(real):
    def wrapper(*a, **kw):
        COUNT[0] += 1
        return real(*a, **kw)
    return wrapper


with mock.patch.object(wp, "Event", LaunchCountEvent), \
     mock.patch.object(wp, "record_event", record), \
     mock.patch.object(wp, "get_event_elapsed_time", lambda b, e: e.n - b.n), \
     mock.patch.object(wp, "launch", counting(wp.launch)), \
     mock.patch.object(wp, "launch_tiled", counting(wp.launch_tiled)):
    with EventTracer() as tracer:
        mjw.step(m, d)
        structure = tracer.trace()
print("event_scope tree of one mjw.step (launches inside each scope, not time):")
show(structure, "launches", 1.0)


def to_json(tr):
    return [{"name": k, "launches": int(sum(v)), "children": to_json(s)} for k, (v, s) in tr.items()]


if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"model": "PLAYGROUND", "tree": to_json(structure)}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

### Trace pretty-printer and icicle chart
Uses `trace` and `structure` from Example 7.10.

In [ ]:
# Icicle chart of the event_scope tree from Example 7.10.
# On a GPU this uses the real EventTracer times (mean ms per call); on CPU it falls back to the
# launch-count structure, so the chart shows "launches inside each scope" instead of time.
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("skipped: matplotlib not installed")

tree = trace if (ON_GPU and trace) else structure
unit = "ms" if (ON_GPU and trace) else "launches"


def boxes(tr, depth=0, x0=0.0, out=None):
    out = [] if out is None else out
    x = x0
    for name, (vals, sub) in tr.items():
        w = sum(vals) / len(vals) if unit == "ms" else sum(vals)
        out.append((name, depth, x, w))
        boxes(sub, depth + 1, x, out)
        x += w
    return out


def show_tree(tr, depth=0):
    for name, (vals, sub) in tr.items():
        v = sum(vals) / len(vals) if unit == "ms" else sum(vals)
        print(f"{'  ' * depth}{name:<{30 - 2 * depth}} {v:10.4f} {unit}")
        show_tree(sub, depth + 1)


show_tree(tree)
if plt is not None:
    bx = boxes(tree)
    fig, ax = plt.subplots(figsize=(12, 0.6 * (1 + max(b[1] for b in bx))))
    for name, depth, x, w in bx:
        if w <= 0:
            continue
        ax.barh(-depth, w, left=x, height=0.9, color=plt.cm.tab20(hash(name) % 20), edgecolor="white")
        if w > 0.04 * bx[0][3]:
            ax.text(x + w / 2, -depth, name, ha="center", va="center", fontsize=8)
    ax.set_yticks([])
    ax.set_xlabel(unit)
    ax.set_title(f"event_scope tree of one mjw.step ({unit})")
    plt.show()

## 11. Part I checkpoint: the bouncing-balls dashboard
10,240 balls on a GPU (64 on CPU), 2 s, bounces counted on the device by a custom kernel. The script prints the summary and saves `ex11_dashboard.png`.

In [ ]:
# Example 7.11 - ex11_checkpoint_dashboard.py
"""Example 7.11 - Part I checkpoint: a dashboard of many bouncing balls.

NWORLD balls (10,240 on a GPU) start at random heights with random velocities and bounce for 2 s.
Everything from Part I comes together: one Model, a batched Data, per-world initial state with
.assign(), mjw.step in a loop, a custom Warp kernel that counts bounces on the device (no .numpy()
in the loop), contact sizing with an overflow check, and a summary plus a 4-panel figure.
The ball model is M1 BALL_DROP with a bouncier contact (solref="0.02 0.1": low damping ratio).
Run: python ex11_checkpoint_dashboard.py [--dump-json PATH] [--dump-png DIR]
  --dump-json PATH  traces, histograms and the height/bounce scatter for the page's dashboard
  --dump-png DIR    also save the 4-panel matplotlib figure as DIR/ex11_dashboard.png
"""
import argparse
import json
import os
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

BOUNCY_BALLS = """
<mujoco model="bouncy_balls">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1" solref="0.02 0.1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1" solref="0.02 0.1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 10240 if ON_GPU else 64
NSTEP = 1000                     # 2 s at timestep 0.002 (same on CPU and GPU)
EVERY = 10                       # keep a height sample every 10 steps (0.02 s)
RADIUS = 0.1


@wp.kernel
def count_bounces(qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
                  prev_vz: wp.array(dtype=float), bounces: wp.array(dtype=int),
                  min_z: wp.array(dtype=float)):
    w = wp.tid()                                      # one thread per world
    z, vz = qpos[w, 2], qvel[w, 2]
    if prev_vz[w] < -0.05 and vz >= 0.0 and z < RADIUS + 0.02:
        bounces[w] += 1                               # falling -> rising, at the floor
    prev_vz[w] = vz
    min_z[w] = wp.min(min_z[w], z)


@wp.kernel
def record_height(qpos: wp.array2d(dtype=float), frame: int, out: wp.array2d(dtype=float)):
    w = wp.tid()
    out[frame, w] = qpos[w, 2]


mjm = mujoco.MjModel.from_xml_string(BOUNCY_BALLS)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=2, njmax=8)    # 1 contact = 4 pyramidal rows

rng = np.random.default_rng(7)
h0 = rng.uniform(0.3, 2.0, NWORLD).astype(np.float32)        # start heights [m]
qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
qpos[:, 2] = h0
qvel = np.zeros((NWORLD, mjm.nv), np.float32)
qvel[:, 0:2] = rng.normal(0.0, 0.5, (NWORLD, 2))             # sideways drift
qvel[:, 2] = rng.uniform(-1.0, 1.0, NWORLD)                  # initial vertical speed
d.qpos.assign(qpos)
d.qvel.assign(qvel)

prev_vz = wp.zeros(NWORLD, dtype=float)
bounces = wp.zeros(NWORLD, dtype=int)
min_z = wp.full(NWORLD, 10.0, dtype=float)
heights = wp.zeros((NSTEP // EVERY, NWORLD), dtype=float)

mjw.step(m, d)                                               # warm-up (JIT) outside the clock
wp.launch(count_bounces, dim=NWORLD, inputs=[d.qpos, d.qvel, prev_vz, bounces, min_z])
wp.synchronize()
t0 = time.perf_counter()
for i in range(1, NSTEP):
    mjw.step(m, d)
    wp.launch(count_bounces, dim=NWORLD, inputs=[d.qpos, d.qvel, prev_vz, bounces, min_z])
    if i % EVERY == 0:
        wp.launch(record_height, dim=NWORLD, inputs=[d.qpos, i // EVERY, heights])
wp.synchronize()
wall = time.perf_counter() - t0

b = bounces.numpy()
zf = d.qpos.numpy()[:, 2]
zmin = min_z.numpy()
H = heights.numpy()
print(f"device: {wp.get_device()} | nworld: {NWORLD} | {NSTEP} steps = "
      f"{NSTEP * mjm.opt.timestep:.1f} s simulated")
print(f"bounces per world: mean {b.mean():.2f}, min {b.min()}, max {b.max()}")
print("bounce count histogram:", {int(k): int(v) for k, v in zip(*np.unique(b, return_counts=True))})
edges = np.array([0.0, 0.11, 0.25, 0.5, 1.0, 3.0])
counts, _ = np.histogram(zf, bins=edges)
print("final z histogram:", {f"{edges[k]:.2f}-{edges[k + 1]:.2f}": int(c) for k, c in enumerate(counts)})
print(f"deepest penetration: {RADIUS - zmin.min():.4f} m (ball radius {RADIUS} m)")
print(f"corr(start height, bounces): {np.corrcoef(h0, b)[0, 1]:.2f}")
print(f"wall time {wall:.2f} s -> {NWORLD * (NSTEP - 1) / wall:,.0f} world-steps/s "
      f"({'GPU' if ON_GPU else 'CPU: not a GPU number'})")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    show = min(NWORLD, 8)
    out = {"nworld": NWORLD, "dt": EVERY * float(mjm.opt.timestep), "device": str(wp.get_device()),
           "traces": [[round(float(v), 3) for v in H[1:, w]] for w in range(show)],
           "bounce_hist": {int(k): int(v) for k, v in zip(*np.unique(b, return_counts=True))},
           "z_edges": edges.tolist(), "z_counts": counts.tolist(),
           "scatter": [[round(float(h0[w]), 3), int(b[w])] for w in range(min(NWORLD, 256))],
           "steps_per_s": round(NWORLD * (NSTEP - 1) / wall)}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("skipped: matplotlib not installed")
if plt is not None:
    fig, ax = plt.subplots(2, 2, figsize=(9, 6.5))
    t = np.arange(1, H.shape[0]) * EVERY * mjm.opt.timestep
    for w in range(min(NWORLD, 8)):
        ax[0, 0].plot(t, H[1:, w], lw=1)
    ax[0, 0].set(title="height of 8 worlds", xlabel="t [s]", ylabel="z [m]")
    ks, vs = np.unique(b, return_counts=True)
    ax[0, 1].bar(ks, vs, color="#4e79a7")
    ax[0, 1].set(title="bounces per world", xlabel="bounces in 2 s", ylabel="worlds")
    ax[1, 0].bar(range(len(counts)), counts, color="#59a14f",
                 tick_label=[f"{edges[k]:.2f}-\n{edges[k + 1]:.2f}" for k in range(len(counts))])
    ax[1, 0].set(title="final height", xlabel="z [m]", ylabel="worlds")
    ax[1, 1].scatter(h0, b + rng.uniform(-0.15, 0.15, NWORLD), s=6, alpha=0.6, color="#f28e2b")
    ax[1, 1].set(title="start height vs bounces", xlabel="h0 [m]", ylabel="bounces (jittered)")
    fig.suptitle(f"{NWORLD} bouncing balls, mujoco-warp {mjw.__version__}, {wp.get_device()}")
    fig.tight_layout()
    path = os.path.join(args.dump_png or ".", "ex11_dashboard.png")
    if args.dump_png:
        os.makedirs(args.dump_png, exist_ok=True)
    fig.savefig(path, dpi=80)
    print("saved", path)

In [ ]:
# Show the 4-panel figure Example 7.11 saved (needs matplotlib, which the setup cell installs).
import os
try:
    from IPython.display import Image, display
except ImportError:                       # outside Jupyter
    Image = display = None
if os.path.exists("ex11_dashboard.png") and display is not None:
    display(Image("ex11_dashboard.png"))
else:
    print("ex11_dashboard.png not found: run Example 7.11 first (and make sure matplotlib is installed)")

## Exercise 7.1: reproduce `forward` with the `_forward` pairing
Five manual calls with the same factorize pairing as `_forward` (not the public defaults). The assert checks your result.

In [ ]:
# Exercise 7.1: reproduce forward with the _forward factorize pairing
import os
HUMANOID_PATH = os.path.join(os.path.dirname(mjw.__file__), "test_data", "humanoid", "humanoid.xml")
mjm = mujoco.MjModel.from_xml_path(HUMANOID_PATH)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=32, njmax=128)
d.ctrl.assign(np.random.default_rng(1).uniform(-1, 1, (4, mjm.nu)).astype(np.float32))
for _ in range(20):
    mjw.step(m, d)
mjw.forward(m, d)
qacc_ref = d.qacc.numpy().copy()
d.qacc.zero_()                      # so that the check below really tests your calls

# your five calls here

err = np.abs(d.qacc.numpy() - qacc_ref).max()
print("max |qacc diff|:", err)
if err < 1e-5:
    print("Exercise 7.1 passed")
else:
    print("not solved yet: qacc does not match forward")

In [ ]:
#@title Exercise 7.1: solution (double-click to show the code)
import os
HUMANOID_PATH = os.path.join(os.path.dirname(mjw.__file__), "test_data", "humanoid", "humanoid.xml")
mjm = mujoco.MjModel.from_xml_path(HUMANOID_PATH)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=32, njmax=128)
d.ctrl.assign(np.random.default_rng(1).uniform(-1, 1, (4, mjm.nu)).astype(np.float32))
for _ in range(20):
    mjw.step(m, d)
mjw.forward(m, d)
qacc_ref = d.qacc.numpy().copy()

mjw.fwd_position(m, d, factorize=False)     # no factor in the position stage ...
mjw.fwd_velocity(m, d)
mjw.fwd_actuation(m, d)
mjw.fwd_acceleration(m, d, factorize=True)  # ... a fused factor + solve here
mjw.solve(m, d)

err = np.abs(d.qacc.numpy() - qacc_ref).max()
print("max |qacc diff|:", err)
assert err < 1e-5, err
print("Exercise 7.1 passed")

## Exercise 7.2: fewer launches without contacts
Write `count_launches(m, d)`, count for the playground, disable `CONTACT`, count again.

In [ ]:
# Exercise 7.2: fewer launches without contacts
from unittest import mock
import mujoco
import numpy as np
import warp as wp
import mujoco_warp as mjw

XML = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
</mujoco>
"""


def count_launches(m, d):
    ...  # your code: wrap wp.launch and wp.launch_tiled, run mjw.step(m, d), return the count


mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=16, njmax=64)
for _ in range(150):
    mjw.step(m, d)
n_default = count_launches(m, d)
m.opt.disableflags = int(mjw.DisableBit.CONTACT)
mjw.step(m, d)                      # warm up the new configuration
n_nocontact = count_launches(m, d)
print("launches: default", n_default, "| CONTACT disabled", n_nocontact)
if n_default is None or n_nocontact is None:
    print("not solved yet: count_launches returns None")
else:
    assert n_nocontact < n_default
    print("Exercise 7.2 passed")
m.opt.disableflags = 0

In [ ]:
#@title Exercise 7.2: solution (double-click to show the code)
def count_launches(m, d):
    n = [0]

    def spy(real):
        def wrapper(*a, **kw):
            n[0] += 1
            return real(*a, **kw)
        return wrapper

    with mock.patch.object(wp, "launch", spy(wp.launch)), \
         mock.patch.object(wp, "launch_tiled", spy(wp.launch_tiled)):
        mjw.step(m, d)
    return n[0]


m.opt.disableflags = 0
d = mjw.make_data(mjm, nworld=4, nconmax=16, njmax=64)
for _ in range(150):
    mjw.step(m, d)
n_default = count_launches(m, d)
m.opt.disableflags = int(mjw.DisableBit.CONTACT)
mjw.step(m, d)
n_nocontact = count_launches(m, d)
m.opt.disableflags = 0
print("launches: default", n_default, "| CONTACT disabled", n_nocontact)
assert n_nocontact < n_default
print("Exercise 7.2 passed")

## Exercise 7.3: audit the humanoid's force budget
Rebuild `qfrc_smooth` from its terms (use `mjw.xfrc_accumulate` for the `J^T xfrc` term).

In [ ]:
# Exercise 7.3: audit the humanoid's force budget
import os
HUMANOID_PATH = os.path.join(os.path.dirname(mjw.__file__), "test_data", "humanoid", "humanoid.xml")
mjm = mujoco.MjModel.from_xml_path(HUMANOID_PATH)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=32, njmax=128)
rng = np.random.default_rng(2)
d.ctrl.assign(rng.uniform(-1, 1, (4, mjm.nu)).astype(np.float32))
xfrc = np.zeros((4, mjm.nbody, 6), np.float32)
xfrc[:, 1, :3] = rng.normal(0, 20, (4, 3))      # push the torso
d.xfrc_applied.assign(xfrc)
for _ in range(10):
    mjw.step(m, d)
mjw.forward(m, d)

jt = None   # J^T xfrc_applied as a (4, nv) wp.array
res = None  # max |qfrc_smooth - (sum of the terms)|

if jt is None or res is None:
    print("not solved yet: fill in jt and res")
else:
    print("max residual:", res)
    assert res < 1e-4
    assert np.abs(jt.numpy()).max() > 1.0
    print("Exercise 7.3 passed")

In [ ]:
#@title Exercise 7.3: solution (double-click to show the code)
import os
HUMANOID_PATH = os.path.join(os.path.dirname(mjw.__file__), "test_data", "humanoid", "humanoid.xml")
mjm = mujoco.MjModel.from_xml_path(HUMANOID_PATH)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=32, njmax=128)
rng = np.random.default_rng(2)
d.ctrl.assign(rng.uniform(-1, 1, (4, mjm.nu)).astype(np.float32))
xfrc = np.zeros((4, mjm.nbody, 6), np.float32)
xfrc[:, 1, :3] = rng.normal(0, 20, (4, 3))
d.xfrc_applied.assign(xfrc)
for _ in range(10):
    mjw.step(m, d)
mjw.forward(m, d)

jt = wp.zeros((4, mjm.nv), dtype=float)
mjw.xfrc_accumulate(m, d, jt)                   # adds J^T xfrc_applied into jt
rhs = (d.qfrc_passive.numpy() - d.qfrc_bias.numpy() + d.qfrc_actuator.numpy()
       + d.qfrc_applied.numpy() + jt.numpy())
res = np.abs(d.qfrc_smooth.numpy() - rhs).max()
print("max residual:", res)
assert res < 1e-4
assert np.abs(jt.numpy()).max() > 1.0
print("Exercise 7.3 passed")

## Stretch: why do the stage shares differ between CPU and GPU?
Rerun Example 7.8 with `NWORLD = 8` and `NWORLD = 4096` (edit the budget block) for both models, compare with the CPU figure on the chapter page, and explain the differences: per-launch overhead, the eager solver loop's device-to-host sync per iteration, and how CPU time grows linearly with `nworld`. Use Example 7.10's tree to find the most expensive scope on your GPU.